# LDA Hyperparameter Tuning

Grid-search hyperparameter tuning for Gensim LDA across subjects (cs, math, physics).
Evaluates **Coherence (C_v)**, **IRBO Diversity**, and **Topic Quality** (harmonic mean).
Best models are saved per subject by **Topic Quality**.

In [ ]:
import os
import gc
import pickle
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product, combinations
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
import warnings
import rbo as rb

warnings.filterwarnings("ignore", category=FutureWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../dataset")
TUNNING_DIR = Path("./tunning")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (TUNNING_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Output directory: {TUNNING_DIR}")

Subjects: ['cs', 'math', 'physics']
Output directory: tunning


## Hyperparameter Grid

In [3]:

PARAM_GRID = {
    "num_topics": [25, 50, 75, 150],   
    "alpha": ["asymmetric", 0.01], 
    "eta": ["auto", 0.01],             
    "passes": [15],                   
}

FIXED_PARAMS = {
    "chunksize": 2000,
    "random_state": 42,
    "workers": 5,
}

keys = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
all_combos = list(product(*values))

print(f"Tunable parameters: {keys}")
print(f"Total parameter combinations: {len(all_combos)}")
print(f"Total runs (combinations x subjects): {len(all_combos) * len(LIST_SUBJECT)}")

Tunable parameters: ['num_topics', 'alpha', 'eta', 'passes']
Total parameter combinations: 16
Total runs (combinations x subjects): 48


## Helper Functions

In [ ]:
def load_and_preprocess(subject: str):
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    
    token_list = df["text"].tolist()
    tokens = [x.split() for x in token_list]

    dictionary = Dictionary(tokens)
    dictionary.filter_extremes(no_below=15, no_above=0.5)
    dictionary.compactify()

    corpus = [dictionary.doc2bow(text) for text in tokens]

    return df, tokens, dictionary, corpus


def train_lda(corpus, dictionary, params: dict) -> LdaMulticore:
    model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=params["num_topics"],
        alpha=params["alpha"],
        eta=params["eta"],
        passes=params["passes"],
        chunksize=FIXED_PARAMS["chunksize"],
        random_state=FIXED_PARAMS["random_state"],
        workers=FIXED_PARAMS["workers"],
    )
    return model


def calculate_coherence(model: LdaMulticore, texts, dictionary: Dictionary) -> float:
    topic_tuples = model.show_topics(
        num_topics=model.num_topics,
        num_words=10,
        formatted=False
    )
    topics = [[word for word, _ in words_probs] for _, words_probs in topic_tuples]

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v',
        processes=5
    )
    return cm.get_coherence()


def get_topic_words_lda(model: LdaMulticore, top_n: int = 10):
    """Extract top-N words for each topic from an LDA model, preserving rank order."""
    topic_tuples = model.show_topics(
        num_topics=model.num_topics,
        num_words=top_n,
        formatted=False
    )
    topics_words = []
    for _, words_probs in topic_tuples:
        words = [word for word, _ in words_probs]
        topics_words.append(words)
    return topics_words


def rbo(list_1, list_2, p=0.9):
    """
    Rank-Biased Overlap (RBO) between two ranked lists.
    Returns similarity score in [0, 1]. Higher = more similar.
    """
    return rb.RankingSimilarity(list_1, list_2).rbo_ext(p=p)


def calculate_irbo(topics_words, p=0.9):
    """
    Calculate mean IRBO (Inverted RBO) diversity across all topic pairs.
    Returns mean_irbo in [0, 1]. Higher = more diverse.
    """
    if len(topics_words) < 2:
        return 0.0
    
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    
    return np.mean(irbo_scores)

## Load & Preprocess All Subjects

In [5]:
all_data = {}
all_tokens = {}
all_dictionaries = {}
all_corpora = {}

for subject in LIST_SUBJECT:
    print(f"Loading {subject}...")
    df, tokens, dictionary, corpus = load_and_preprocess(subject)
    all_data[subject] = df
    all_tokens[subject] = tokens
    all_dictionaries[subject] = dictionary
    all_corpora[subject] = corpus
    print(f"  {subject}: {len(df):,} documents, {len(dictionary):,} terms in dictionary")

print(f"\nSubjects ready: {list(all_data.keys())}")

Loading cs...
  cs: 165,756 documents, 21,694 terms in dictionary
Loading math...
  math: 157,085 documents, 16,835 terms in dictionary
Loading physics...
  physics: 146,311 documents, 21,894 terms in dictionary

Subjects ready: ['cs', 'math', 'physics']


## Hyperparameter Tuning Grid Search

For each parameter combination, compute **Coherence**, **IRBO Diversity**, and **Topic Quality** (harmonic mean).
Best models are saved per subject by Topic Quality.

In [6]:
total_runs = len(all_combos) * len(LIST_SUBJECT)
run_counter = 0

for subject in LIST_SUBJECT:
    corpus = all_corpora[subject]
    dictionary = all_dictionaries[subject]
    tokens = all_tokens[subject]
    n_docs = len(all_data[subject])

    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    best_model_path = TUNNING_DIR / subject / "best_model.pkl"
    best_quality = -1.0

    existing_results = []
    if results_csv_path.exists():
        existing_df = pd.read_csv(results_csv_path)
        existing_results = existing_df.to_dict('records')
        if len(existing_results) > 0:
            if 'topic_quality' in existing_df.columns:
                best_quality = existing_df["topic_quality"].max()
            else:
                best_quality = existing_df["coherence_cv"].max()
            print(f"Resuming {subject}: {len(existing_results)} previous runs found, best quality so far: {best_quality:.4f}")

    results = existing_results.copy()

    completed_param_sets = set()
    for r in existing_results:
        param_key = (r["num_topics"], str(r["alpha"]), str(r["eta"]), r["passes"])
        completed_param_sets.add(param_key)

    print(f"\n{'='*70}")
    print(f"Subject: {subject.upper()} ({n_docs:,} documents)")
    print(f"{'='*70}")

    for combo in all_combos:
        run_counter += 1
        params = dict(zip(keys, combo))

        param_key = (params["num_topics"], str(params["alpha"]), str(params["eta"]), params["passes"])
        if param_key in completed_param_sets:
            continue

        print(f"\n[{run_counter}/{total_runs}] {subject} | "
              f"k={params['num_topics']} α={params['alpha']} η={params['eta']} "
              f"passes={params['passes']}")

        try:
            start_time = time.time()

            model = train_lda(corpus, dictionary, params)
            coherence = calculate_coherence(model, tokens, dictionary)

            # Compute IRBO diversity
            topics_words = get_topic_words_lda(model, top_n=TOP_N_WORDS)
            irbo_mean = calculate_irbo(topics_words, p=RBO_P)

            # Topic Quality = harmonic mean of coherence and IRBO
            if coherence + irbo_mean > 0:
                topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
            else:
                topic_quality = 0.0

            elapsed = time.time() - start_time

            result_row = {
                "subject": subject,
                "num_topics": params["num_topics"],
                "alpha": str(params["alpha"]),
                "eta": str(params["eta"]),
                "passes": params["passes"],
                "chunksize": FIXED_PARAMS["chunksize"],
                "random_state": FIXED_PARAMS["random_state"],
                "workers": FIXED_PARAMS["workers"],
                "coherence_cv": coherence,
                "irbo_mean": irbo_mean,
                "topic_quality": topic_quality,
                "time_seconds": round(elapsed, 1),
            }
            results.append(result_row)

            is_new_best = topic_quality > best_quality
            if is_new_best:
                best_quality = topic_quality
                with open(best_model_path, "wb") as f:
                    pickle.dump(model, f)
                print(f"  ⭐ NEW BEST | Quality: {topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f}) "
                      f"({elapsed:.1f}s) → Saved to {best_model_path}")
            else:
                print(f"  ✓ Quality: {topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f}) "
                      f"({elapsed:.1f}s) | Best: {best_quality:.4f}")

            pd.DataFrame(results).to_csv(results_csv_path, index=False)

            del model
            gc.collect()

        except Exception as e:
            print(f"  ✗ ERROR: {e}")
            result_row = {
                "subject": subject,
                "num_topics": params["num_topics"],
                "alpha": str(params["alpha"]),
                "eta": str(params["eta"]),
                "passes": params["passes"],
                "chunksize": FIXED_PARAMS["chunksize"],
                "random_state": FIXED_PARAMS["random_state"],
                "workers": FIXED_PARAMS["workers"],
                "coherence_cv": None,
                "irbo_mean": None,
                "topic_quality": None,
                "time_seconds": None,
            }
            results.append(result_row)
            pd.DataFrame(results).to_csv(results_csv_path, index=False)
            gc.collect()

    print(f"\n{'='*70}")
    print(f"✅ {subject.upper()} COMPLETE | Best quality: {best_quality:.4f}")
    print(f"Results saved to: {results_csv_path}")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*70}")


Subject: CS (165,756 documents)

[1/48] cs | k=25 α=asymmetric η=auto passes=15
  ⭐ NEW BEST | Quality: 0.6822 (C=0.5218, IRBO=0.9852) (193.4s) → Saved to tunning/cs/best_model.pkl

[2/48] cs | k=25 α=asymmetric η=0.01 passes=15
  ⭐ NEW BEST | Quality: 0.6822 (C=0.5219, IRBO=0.9848) (223.1s) → Saved to tunning/cs/best_model.pkl

[3/48] cs | k=25 α=0.01 η=auto passes=15
  ✓ Quality: 0.6764 (C=0.5158, IRBO=0.9824) (194.9s) | Best: 0.6822

[4/48] cs | k=25 α=0.01 η=0.01 passes=15
  ✓ Quality: 0.6790 (C=0.5185, IRBO=0.9833) (218.1s) | Best: 0.6822

[5/48] cs | k=50 α=asymmetric η=auto passes=15
  ⭐ NEW BEST | Quality: 0.6972 (C=0.5374, IRBO=0.9926) (406.4s) → Saved to tunning/cs/best_model.pkl

[6/48] cs | k=50 α=asymmetric η=0.01 passes=15
  ⭐ NEW BEST | Quality: 0.6973 (C=0.5375, IRBO=0.9922) (413.7s) → Saved to tunning/cs/best_model.pkl

[7/48] cs | k=50 α=0.01 η=auto passes=15
  ⭐ NEW BEST | Quality: 0.6979 (C=0.5382, IRBO=0.9926) (255.8s) → Saved to tunning/cs/best_model.pkl

[8/48] 

## Summary: Best Parameters Per Subject (by Topic Quality)

In [7]:
print("\n" + "=" * 110)
print("FINAL RESULTS: Best Parameters Per Subject (by Topic Quality)")
print("=" * 110)

for subject in LIST_SUBJECT:
    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    if not results_csv_path.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    df = pd.read_csv(results_csv_path)
    df_valid = df.dropna(subset=["coherence_cv"])

    if len(df_valid) == 0:
        print(f"\n{subject.upper()}: No valid results")
        continue

    best_row = df_valid.loc[df_valid["topic_quality"].idxmax()]

    print(f"\n{'─'*50}")
    print(f"  Subject:       {subject.upper()}")
    print(f"  Total runs:    {len(df_valid)}")
    print(f"  Best Quality:  {best_row['topic_quality']:.4f}")
    print(f"  Coherence:     {best_row['coherence_cv']:.4f}")
    print(f"  IRBO:          {best_row['irbo_mean']:.4f}")
    print(f"  Parameters:")
    print(f"    num_topics = {best_row['num_topics']}")
    print(f"    alpha      = {best_row['alpha']}")
    print(f"    eta        = {best_row['eta']}")
    print(f"    passes     = {best_row['passes']}")
    print(f"    chunksize  = {best_row['chunksize']}")
    print(f"{'─'*50}")

print("\n" + "=" * 110)
print("Top 5 configurations per subject (by Topic Quality):")
print("=" * 110)

for subject in LIST_SUBJECT:
    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    if not results_csv_path.exists():
        continue

    df = pd.read_csv(results_csv_path)
    df_valid = df.dropna(subset=["coherence_cv"]).sort_values("topic_quality", ascending=False)

    print(f"\n{subject.upper()}:")
    print(df_valid[["num_topics", "alpha", "eta", "passes", "coherence_cv",
                    "irbo_mean", "topic_quality", "time_seconds"]].head(5).to_string(index=False))
    print()


FINAL RESULTS: Best Parameters Per Subject (by Topic Quality)

──────────────────────────────────────────────────
  Subject:       CS
  Total runs:    16
  Best Quality:  0.7130
  Coherence:     0.5550
  IRBO:          0.9967
  Parameters:
    num_topics = 75
    alpha      = asymmetric
    eta        = 0.01
    passes     = 15
    chunksize  = 2000
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  Subject:       MATH
  Total runs:    16
  Best Quality:  0.7203
  Coherence:     0.5653
  IRBO:          0.9924
  Parameters:
    num_topics = 50
    alpha      = asymmetric
    eta        = auto
    passes     = 15
    chunksize  = 2000
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  Subject:       PHYSICS
  Total runs:    16
  Best Quality:  0.7269
  Coherence:     0.5731
  IRBO:          0.9934
  Parameters:
    num_topics = 50
    alpha      = asymmetric
    eta        = 0.01
